### Iniciar Spark

In [12]:
# ============================================================
# INICIALIZAR SPARK Y SILENCIAR LOGS (VERSIÓN MEJORADA)
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, desc, when, row_number, rank, dense_rank, datediff, year, month, hour
from pyspark.sql.functions import round as spark_round
import logging
import warnings
import os

# 1. CREAR SESIÓN DE SPARK CON CONFIGURACIONES ESTABLES
print("🚀 Inicializando Spark...")

spark = SparkSession.builder \
    .appName("PagilaAnalysis") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.sql.broadcastTimeout", "1200") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m")\
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")\
    .config("spark.sql.adaptive.coalescePartitions.enabled", "false")\
    .config("spark.sql.adaptive.skewJoin.enabled", "false")\
    .config("spark.ui.enabled", "false")\
    .getOrCreate()

print("✅ Sesión de Spark creada exitosamente")
print(f"  - Versión: {spark.version}")
print(f"  - Aplicación: {spark.sparkContext.appName}")

# 2. Desactivar logs de Spark
spark.sparkContext.setLogLevel("ERROR")
print("✅ Logs de Spark silenciados (nivel ERROR)")

# 3. Configurar logging de Python
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("org.apache.spark").setLevel(logging.ERROR)
logging.getLogger("org.apache.parquet").setLevel(logging.ERROR)

# 4. Silenciar pandas warnings
warnings.filterwarnings('ignore')

# 5. Configurar variables de entorno para evitar errores de conexión
os.environ["PYSPARK_PIN_THREAD"] = "true"

print("✅ Configuración completa aplicada")
print("\n📋 Configuraciones aplicadas:")
print("  - Adaptive Query Execution: Desactivado")
print("  - Shuffle Partitions: 4")
print("  - Executor Memory: 2g")
print("  - Driver Memory: 2g")
print("  - Network Timeout: 800s")
print("  - Kryo Serializer: Activado")

🚀 Inicializando Spark...
✅ Sesión de Spark creada exitosamente
  - Versión: 4.1.2
  - Aplicación: PagilaAnalysis
✅ Logs de Spark silenciados (nivel ERROR)
✅ Configuración completa aplicada

📋 Configuraciones aplicadas:
  - Adaptive Query Execution: Desactivado
  - Shuffle Partitions: 4
  - Executor Memory: 2g
  - Driver Memory: 2g
  - Network Timeout: 800s
  - Kryo Serializer: Activado


### 1. Cargar datos y conectar sql

In [13]:
# ============================================================
# RECARGAR DATOS CON TIPOS CORRECTOS (DESDE CERO)
# ============================================================

import psycopg2
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

print("="*60)
print("🔄 RECARGANDO DATOS CON TIPOS CORRECTOS")
print("="*60)

# ------------------------------------------------------------
# 1. CONECTAR A POSTGRES
# ------------------------------------------------------------
print("\n🔌 Conectando a PostgreSQL...")
conn = psycopg2.connect(
    host="172.17.0.2",
    database="pagila",
    user="admin",
    password="admin123"
)
print("✅ Conexión exitosa")

# ------------------------------------------------------------
# 2. FUNCIÓN PARA CARGAR CON TIPOS EXPLÍCITOS
# ------------------------------------------------------------
def cargar_con_tipos(nombre, query, columnas_numericas):
    """
    Carga una tabla de PostgreSQL a Spark DataFrame con tipos correctos.
    - columnas_numericas: lista de columnas que deben ser numéricas (int o double)
    - Las demás columnas se mantienen como string
    """
    print(f"  Cargando {nombre}...")
    
    # Leer con pandas
    df_pandas = pd.read_sql(query, conn)
    
    # Reemplazar None por NULL (0) para columnas numéricas
    for col in columnas_numericas:
        if col in df_pandas.columns:
            # Reemplazar None por 0 en columnas numéricas
            df_pandas[col] = df_pandas[col].fillna(0)
            # Convertir a numérico (int o float)
            try:
                df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce').fillna(0).astype(int)
            except:
                df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce').fillna(0.0).astype(float)
    
    # Convertir a Spark (sin astype(str) para evitar 'None')
    df_spark = spark.createDataFrame(df_pandas)
    
    print(f"  ✅ {nombre}: {df_spark.count():,} registros")
    return df_spark



# ------------------------------------------------------------
# 3. DEFINIR COLUMNAS NUMÉRICAS POR TABLA
# ------------------------------------------------------------
columnas_numericas = {
    'film': ['film_id', 'release_year', 'language_id', 'original_language_id', 
             'rental_duration', 'rental_rate', 'length', 'replacement_cost'],
    'actor': ['actor_id'],
    'film_actor': ['actor_id', 'film_id'],
    'category': ['category_id'],
    'customer': ['customer_id', 'store_id', 'address_id', 'active', 'create_date'],
    'rental': ['rental_id', 'customer_id', 'inventory_id', 'staff_id'],
    'payment': ['payment_id', 'customer_id', 'staff_id', 'rental_id'],
    'inventory': ['inventory_id', 'film_id', 'store_id'],
    'film_category': ['film_id', 'category_id'],
    'address': ['address_id', 'city_id', 'postal_code'],  # ¡NUEVA!
    'city': ['city_id', 'country_id'],  # ¡NUEVA!
    'country': ['country_id']  # ¡NUEVA!
}

# ------------------------------------------------------------
# 4. CARGAR TABLAS
# ------------------------------------------------------------
print("\n📥 Cargando tablas con tipos correctos...")

film = cargar_con_tipos('film', 'SELECT * FROM film', columnas_numericas['film'])
actor = cargar_con_tipos('actor', 'SELECT * FROM actor', columnas_numericas['actor'])
category = cargar_con_tipos('category', 'SELECT * FROM category', columnas_numericas['category'])
customer = cargar_con_tipos('customer', 'SELECT * FROM customer', columnas_numericas['customer'])
rental = cargar_con_tipos('rental', 'SELECT * FROM rental', columnas_numericas['rental'])
payment = cargar_con_tipos('payment', 'SELECT * FROM payment', columnas_numericas['payment'])
inventory = cargar_con_tipos('inventory', 'SELECT * FROM inventory', columnas_numericas['inventory'])
film_category = cargar_con_tipos('film_category', 'SELECT * FROM film_category', columnas_numericas['film_category'])
film_actor = cargar_con_tipos('film_actor', 'SELECT * FROM film_actor', columnas_numericas['film_actor'])
address = cargar_con_tipos('address', 'SELECT * FROM address', columnas_numericas['address'])
city = cargar_con_tipos('city', 'SELECT * FROM city', columnas_numericas['city'])
country = cargar_con_tipos('country', 'SELECT * FROM country', columnas_numericas['country'])

conn.close()
print("\n✅ Todas las tablas cargadas")

# ------------------------------------------------------------
# 5. VERIFICAR ESQUEMAS
# ------------------------------------------------------------
print("\n📋 Esquema de film (corregido):")
film.printSchema()

print("\n📄 Muestra de film:")
film.select("film_id", "title", "rental_rate", "length", "rating").show(5)

print("\n✅ ¡Ahora sí funcionan los análisis!")

🔄 RECARGANDO DATOS CON TIPOS CORRECTOS

🔌 Conectando a PostgreSQL...
✅ Conexión exitosa

📥 Cargando tablas con tipos correctos...
  Cargando film...
  ✅ film: 1,000 registros
  Cargando actor...
  ✅ actor: 200 registros
  Cargando category...
  ✅ category: 16 registros
  Cargando customer...
  ✅ customer: 600 registros
  Cargando rental...
  ✅ rental: 16,044 registros
  Cargando payment...
  ✅ payment: 16,049 registros
  Cargando inventory...
  ✅ inventory: 4,581 registros
  Cargando film_category...
  ✅ film_category: 2,367 registros
  Cargando film_actor...
  ✅ film_actor: 5,462 registros
  Cargando address...
  ✅ address: 603 registros
  Cargando city...
  ✅ city: 600 registros
  Cargando country...
  ✅ country: 109 registros

✅ Todas las tablas cargadas

📋 Esquema de film (corregido):
root
 |-- film_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: long (nullable = true)
 |-- language_id: long (nullable = t

Traceback (most recent call last):
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


### 2. Analisis datos exploracion

In [3]:
# ============================================================
# ANÁLISIS COMPLETO CORREGIDO
# ============================================================

print("="*60)
print("🎬 ANÁLISIS DE DATOS - PAGILA (CORREGIDO)")
print("="*60)

from pyspark.sql.functions import col, count, sum as spark_sum, avg, desc, when, row_number, rank, dense_rank, datediff, year, month
from pyspark.sql.window import Window

# 1. TOP 10 PELÍCULAS MÁS ALQUILADAS
print("\n🎯 TOP 10 PELÍCULAS MÁS ALQUILADAS:")
print("-"*40)

top_films = film.join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .groupBy("film_id", "title") \
    .agg(count("*").alias("total_alquileres")) \
    .orderBy(desc("total_alquileres")) \
    .limit(10)

top_films.show(truncate=False)

# 2. INGRESOS POR CATEGORÍA
print("\n💰 INGRESOS POR CATEGORÍA:")
print("-"*40)

revenue_by_category = film.join(film_category, "film_id") \
    .join(category, film_category.category_id == category.category_id) \
    .join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .join(payment, "rental_id") \
    .groupBy(category.name.alias("categoria")) \
    .agg(
        spark_sum("amount").alias("ingresos_totales"),
        avg("amount").alias("promedio_por_renta"),
        count("*").alias("total_rentas")
    ) \
    .orderBy(desc("ingresos_totales"))

revenue_by_category.show(truncate=False)

# 3. TOP 5 CLIENTES QUE MÁS GASTAN (CORREGIDO)
print("\n👑 TOP 5 CLIENTES QUE MÁS GASTAN:")
print("-"*40)

# Especificar qué tabla usar para customer_id
top_customers = customer.join(rental, customer.customer_id == rental.customer_id) \
    .join(payment, rental.rental_id == payment.rental_id) \
    .groupBy(customer.customer_id, customer.first_name, customer.last_name, customer.email) \
    .agg(
        spark_sum(payment.amount).alias("total_gastado"),
        count("*").alias("total_rentas"),
        avg(payment.amount).alias("promedio_renta")
    ) \
    .orderBy(desc("total_gastado")) \
    .limit(5)

top_customers.show(truncate=False)

# 4. RENTABILIDAD POR RATING
print("\n⭐ RENTABILIDAD POR RATING:")
print("-"*40)

rating_analysis = film.join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .join(payment, "rental_id") \
    .groupBy("rating") \
    .agg(
        spark_sum("amount").alias("ingresos_totales"),
        avg("amount").alias("promedio_por_renta"),
        count("*").alias("total_rentas")
    ) \
    .orderBy(desc("ingresos_totales"))

rating_analysis.show(truncate=False)

# 5. CASE WHEN - Clasificación de películas
print("\n📌 CASE WHEN - Clasificación por duración:")
print("-"*40)

peliculas_clasificadas = film.select(
    "title",
    "length",
    when(col("length") < 60, "Corta")
    .when(col("length") < 120, "Media")
    .otherwise("Larga")
    .alias("duracion_clasificacion")
).limit(10)

peliculas_clasificadas.show(truncate=False)

# 6. WINDOW FUNCTION - Ranking de películas
print("\n📌 WINDOW FUNCTION - Ranking de películas:")
print("-"*40)

window_rank = Window.orderBy(desc("total_alquileres"))

peliculas_rank = film.join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .groupBy("film_id", "title") \
    .agg(count("*").alias("total_alquileres")) \
    .withColumn("ranking", rank().over(window_rank)) \
    .withColumn("dense_ranking", dense_rank().over(window_rank)) \
    .filter(col("ranking") <= 10) \
    .select("title", "total_alquileres", "ranking", "dense_ranking")

peliculas_rank.show(truncate=False)

print("\n" + "="*60)
print("✅ ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*60)

🎬 ANÁLISIS DE DATOS - PAGILA (CORREGIDO)

🎯 TOP 10 PELÍCULAS MÁS ALQUILADAS:
----------------------------------------


+-------+-------------------+----------------+
|film_id|title              |total_alquileres|
+-------+-------------------+----------------+
|103    |BUCKET BROTHERHOOD |34              |
|738    |ROCKETEER MOTHER   |33              |
|331    |FORWARD TEMPLE     |32              |
|382    |GRIT CLOCKWORK     |32              |
|730    |RIDGEMONT SUBMARINE|32              |
|489    |JUGGLER HARDLY     |32              |
|767    |SCALAWAG DUCK      |32              |
|621    |NETWORK PEAK       |31              |
|735    |ROBBERS JOON       |31              |
|418    |HOBBIT ALIEN       |31              |
+-------+-------------------+----------------+


💰 INGRESOS POR CATEGORÍA:
----------------------------------------


+-----------+------------------+------------------+------------+
|categoria  |ingresos_totales  |promedio_por_renta|total_rentas|
+-----------+------------------+------------------+------------+
|Foreign    |10507.669999999582|4.318812166049972 |2433        |
|Children   |10437.049999999579|4.356030884807837 |2396        |
|Animation  |10369.549999999608|4.425757575757409 |2343        |
|Documentary|10307.289999999573|4.167929640113051 |2473        |
|Action     |10289.599999999593|4.401026518391614 |2338        |
|Music      |10188.80999999958 |4.211992558908466 |2419        |
|Sci-Fi     |10054.099999999577|4.037791164658464 |2490        |
|New        |9915.23999999957  |4.0077768795471185|2474        |
|Sports     |9902.859999999613 |4.27769330453547  |2315        |
|Games      |9848.229999999605 |4.150117994100128 |2373        |
|Horror     |9807.689999999617 |4.396095921111438 |2231        |
|Travel     |9793.979999999587 |4.077427144046456 |2402        |
|Classics   |9708.7699999

Traceback (most recent call last):
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


+-------------------+----------------+-------+-------------+
|title              |total_alquileres|ranking|dense_ranking|
+-------------------+----------------+-------+-------------+
|BUCKET BROTHERHOOD |34              |1      |1            |
|ROCKETEER MOTHER   |33              |2      |2            |
|GRIT CLOCKWORK     |32              |3      |3            |
|JUGGLER HARDLY     |32              |3      |3            |
|FORWARD TEMPLE     |32              |3      |3            |
|RIDGEMONT SUBMARINE|32              |3      |3            |
|SCALAWAG DUCK      |32              |3      |3            |
|APACHE DIVINE      |31              |8      |4            |
|HOBBIT ALIEN       |31              |8      |4            |
|GOODFELLAS SALUTE  |31              |8      |4            |
|NETWORK PEAK       |31              |8      |4            |
|ROBBERS JOON       |31              |8      |4            |
|RUSH GOODFELLAS    |31              |8      |4            |
|TIMBERLAND SKY     |31 

### 3. Ejercicios practica example

#### 3.1 JOINS MÚLTIPLES

```
alquiler_completo = rental.join(customer, rental.customer_id == customer.customer_id) 

python
    .select(
        rental.rental_id,        # ID del alquiler
        customer.first_name,     # Nombre del cliente
        customer.last_name,      # Apellido del cliente
        film.title,              # Título de la película
        rental.rental_date,      # Fecha de alquiler
        rental.return_date       # Fecha de devolución
    ) 
```

In [4]:

print("\n" + "="*60)
print("📌 EJERCICIO 1: JOINS MÚLTIPLES")
print("="*60)

# Información completa de alquileres
alquiler_completo = rental.join(customer, rental.customer_id == customer.customer_id) \
    .join(inventory, "inventory_id") \
    .join(film, "film_id") \
    .select(
        rental.rental_id,
        customer.first_name,
        customer.last_name,
        film.title,
        rental.rental_date,
        rental.return_date
    ) \
    .limit(5)

alquiler_completo.show(truncate=False)


📌 EJERCICIO 1: JOINS MÚLTIPLES


+---------+----------+-----------+------------------+-------------------+-------------------+
|rental_id|first_name|last_name  |title             |rental_date        |return_date        |
+---------+----------+-----------+------------------+-------------------+-------------------+
|2        |TOMMY     |COLLAZO    |FREAKY POCUS      |2022-05-24 16:54:33|2022-05-28 13:40:33|
|6        |NELSON    |CHRISTENSON|MYSTIC TRUMAN     |2022-05-24 17:08:07|2022-05-26 19:32:07|
|7        |CASSANDRA |WALTERS    |SWARM GOLD        |2022-05-24 17:11:53|2022-05-29 14:34:53|
|13       |RAYMOND   |MCWHORTER  |KING EVOLUTION    |2022-05-24 18:22:55|2022-05-29 22:28:55|
|14       |THEODORE  |CULP       |MONTEREY LABYRINTH|2022-05-24 18:31:15|2022-05-25 20:56:15|
+---------+----------+-----------+------------------+-------------------+-------------------+



#### 3.2 AGREGACIONES MÚLTIPLES

Calcula 5 estadísticas diferentes para cada categoría de películas en un solo comando.
```
--join
film.join(film_category, "film_id") \
    .join(category, "category_id")
--groupby
.groupBy(category.name.alias("categoria"))
--calcular kpis
.agg(
    count("*").alias("total_peliculas"),      # 1. Cuántas películas hay
    avg("rental_rate").alias("precio_promedio"), # 2. Precio promedio
    avg("length").alias("duracion_promedio"),    # 3. Duración promedio
    max("rental_rate").alias("precio_maximo"),   # 4. Precio más caro
    min("rental_rate").alias("precio_minimo")    # 5. Precio más barato
)
```


In [5]:

print("\n" + "="*60)
print("📌 EJERCICIO 2: AGREGACIONES MÚLTIPLES")
print("="*60)

from pyspark.sql.functions import count, avg, max, min, desc, col

# Convertir columnas problemáticas (si son string)
film = film.withColumn("rental_rate", col("rental_rate").cast("double"))
film = film.withColumn("length", col("length").cast("double"))

# Análisis
estadisticas = film.join(film_category, "film_id") \
    .join(category, "category_id") \
    .groupBy(category.name.alias("categoria")) \
    .agg(
        count("*").alias("total_peliculas"),
        avg("rental_rate").alias("precio_promedio"),
        avg("length").alias("duracion_promedio"),
        max("rental_rate").alias("precio_maximo"),
        min("rental_rate").alias("precio_minimo")
    ) \
    .orderBy(desc("total_peliculas")) \
    .limit(5)

estadisticas.show(truncate=False)

# Mostrar resultados bonitos
print("\n📊 TOP 5 CATEGORÍAS CON MÁS PELÍCULAS:")
for row in estadisticas.collect():
    print(f"\n  🎬 {row['categoria']}: {row['total_peliculas']} películas")
    print(f"     Precio promedio: ${row['precio_promedio']:.2f}")
    print(f"     Duración promedio: {row['duracion_promedio']:.0f} min")


📌 EJERCICIO 2: AGREGACIONES MÚLTIPLES
+---------+---------------+------------------+------------------+-------------+-------------+
|categoria|total_peliculas|precio_promedio   |duracion_promedio |precio_maximo|precio_minimo|
+---------+---------------+------------------+------------------+-------------+-------------+
|Music    |152            |2.0526315789473686|120.49342105263158|4.0          |0.0          |
|Drama    |152            |1.855263157894737 |116.375           |4.0          |0.0          |
|Travel   |151            |1.8940397350993377|116.6092715231788 |4.0          |0.0          |
|Children |150            |2.0533333333333332|111.83333333333333|4.0          |0.0          |
|Games    |150            |1.9466666666666668|116.28666666666666|4.0          |0.0          |
+---------+---------------+------------------+------------------+-------------+-------------+


📊 TOP 5 CATEGORÍAS CON MÁS PELÍCULAS:

  🎬 Music: 152 películas
     Precio promedio: $2.05
     Duración promedi

#### 3.3 CASE WHEN CON GROUP BY

```
--Join:
customer.join(rental, customer.customer_id == rental.customer_id) \
    .join(payment, rental.rental_id == payment.rental_id)
--agroupar por cliente:
.groupBy(customer.customer_id, customer.first_name, customer.last_name)
--Calcular gasto:
.agg(spark_sum(payment.amount).alias("total_gastado"))
--Clasificar case when
.withColumn(
    "nivel",
    when(col("total_gastado") < 50, "Bronce")
    .when(col("total_gastado") < 100, "Plata")
    .when(col("total_gastado") < 200, "Oro")
    .otherwise("VIP")
)
```

In [6]:
print("\n" + "="*60)
print("📌 EJERCICIO 3: CASE WHEN CON GROUP BY")
print("="*60)

# Clasificación de clientes por nivel de gasto
nivel_clientes = customer.join(rental, customer.customer_id == rental.customer_id) \
    .join(payment, rental.rental_id == payment.rental_id) \
    .groupBy(customer.customer_id, customer.first_name, customer.last_name) \
    .agg(spark_sum(payment.amount).alias("total_gastado")) \
    .withColumn(
        "nivel",
        when(col("total_gastado") < 50, "Bronce")
        .when(col("total_gastado") < 100, "Plata")
        .when(col("total_gastado") < 200, "Oro")
        .otherwise("VIP")
    ) \
    .orderBy(desc("total_gastado")) \
    .limit(10)

nivel_clientes.show(truncate=False)


📌 EJERCICIO 3: CASE WHEN CON GROUP BY
+-----------+----------+---------+------------------+-----+
|customer_id|first_name|last_name|total_gastado     |nivel|
+-----------+----------+---------+------------------+-----+
|526        |KARL      |SEAL     |221.55000000000013|VIP  |
|148        |ELEANOR   |HUNT     |216.5400000000001 |VIP  |
|144        |CLARA     |SHAW     |195.58000000000007|Oro  |
|137        |RHONDA    |KENNEDY  |194.61000000000007|Oro  |
|178        |MARION    |SNYDER   |194.61000000000004|Oro  |
|459        |TOMMY     |COLLAZO  |186.62000000000012|Oro  |
|469        |WESLEY    |BULL     |177.60000000000002|Oro  |
|468        |TIM       |CARY     |175.61000000000004|Oro  |
|236        |MARCIA    |DEAN     |175.58000000000004|Oro  |
|181        |ANA       |BRADLEY  |174.66000000000005|Oro  |
+-----------+----------+---------+------------------+-----+



#### 3.4 WINDOW FUNCTIONS AVANZADAS

```
Encuentra las 3 películas más taquilleras (que más ingresos generan) dentro de CADA categoría.

--definir ventana
window_categoria = Window.partitionBy("categoria").orderBy(desc("ingresos"))
--joins
film.join(film_category, "film_id") \
    .join(category, "category_id") \
    .join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .join(payment, "rental_id")
--agrupar y calcular ingresos
.groupBy(category.name.alias("categoria"), film.title) \
.agg(spark_sum(payment.amount).alias("ingresos"))
--asignar rangos window function
.withColumn("ranking", row_number().over(window_categoria))
--filtrar y ordenar
.filter(col("ranking") <= 3) \
.orderBy("categoria", "ranking")
```

In [7]:
print("\n" + "="*60)
print("📌 EJERCICIO 4: WINDOW FUNCTIONS AVANZADAS")
print("="*60)

# Ranking de películas por categoría
window_categoria = Window.partitionBy("categoria").orderBy(desc("ingresos"))

ranking_categoria = film.join(film_category, "film_id") \
    .join(category, "category_id") \
    .join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .join(payment, "rental_id") \
    .groupBy(category.name.alias("categoria"), film.title) \
    .agg(spark_sum(payment.amount).alias("ingresos")) \
    .withColumn("ranking", row_number().over(window_categoria)) \
    .filter(col("ranking") <= 3) \
    .orderBy("categoria", "ranking")

ranking_categoria.show(truncate=False)


📌 EJERCICIO 4: WINDOW FUNCTIONS AVANZADAS


+-----------+------------------+------------------+-------+
|categoria  |title             |ingresos          |ranking|
+-----------+------------------+------------------+-------+
|Action     |GOODFELLAS SALUTE |209.69000000000008|1      |
|Action     |HARRY IDAHO       |195.70000000000005|2      |
|Action     |HUSTLER PARTY     |190.78000000000003|3      |
|Animation  |INNOCENT USUAL    |191.74000000000004|1      |
|Animation  |RANGE MOONWALKER  |179.73000000000005|2      |
|Animation  |WITCHES PANIC     |173.70000000000005|3      |
|Children   |SATURDAY LAMBS    |204.72000000000006|1      |
|Children   |TITANS JERK       |201.71000000000006|2      |
|Children   |DOGMA FAMILY      |178.70000000000005|3      |
|Classics   |HARRY IDAHO       |195.70000000000005|1      |
|Classics   |MASSACRE USUAL    |179.70000000000005|2      |
|Classics   |NIGHTMARE CHILL   |169.75            |3      |
|Comedy     |TELEGRAPH VOYAGE  |231.73000000000008|1      |
|Comedy     |ENEMY ODDS        |180.7100

#### 3.5  OPERACIONES CON FECHAS

```
Analiza patrones temporales de alquileres usando funciones de fecha:
tendencias mensuales, duración promedio y películas que la gente retiene más tiempo.
--Alquileres por mes y año
alquileres_mes = rental \
    .withColumn("mes", month("rental_date")) \
    .withColumn("anio", year("rental_date")) \
    .groupBy("anio", "mes") \
    .agg(count("*").alias("total_alquileres")) \
    .orderBy("anio", "mes")
--Duración promedio de alquiler
duracion_promedio = rental \
    .withColumn("dias", datediff("return_date", "rental_date")) \
    .agg(avg("dias").alias("dias_promedio"))
--Top 10 películas con mayor duración de alquiler
peliculas_largas = film.join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .withColumn("dias", datediff("return_date", "rental_date")) \
    .groupBy("title") \
    .agg(avg("dias").alias("dias_promedio")) \
    .orderBy(desc("dias_promedio")) \
    .limit(10)

```

In [8]:
print("\n" + "="*60)
print("📌 EJERCICIO 5: OPERACIONES CON FECHAS")
print("="*60)

# Alquileres por mes y año
alquileres_mes = rental \
    .withColumn("mes", month("rental_date")) \
    .withColumn("anio", year("rental_date")) \
    .groupBy("anio", "mes") \
    .agg(count("*").alias("total_alquileres")) \
    .orderBy("anio", "mes")

alquileres_mes.show()

# Duración promedio de alquiler
duracion_promedio = rental \
    .withColumn("dias", datediff("return_date", "rental_date")) \
    .agg(avg("dias").alias("dias_promedio"))

duracion_promedio.show()

# Top 10 películas con mayor duración de alquiler
peliculas_largas = film.join(inventory, "film_id") \
    .join(rental, "inventory_id") \
    .withColumn("dias", datediff("return_date", "rental_date")) \
    .groupBy("title") \
    .agg(avg("dias").alias("dias_promedio")) \
    .orderBy(desc("dias_promedio")) \
    .limit(10)

peliculas_largas.show(truncate=False)


📌 EJERCICIO 5: OPERACIONES CON FECHAS
+----+---+----------------+
|anio|mes|total_alquileres|
+----+---+----------------+
|2022|  2|             182|
|2022|  5|            1156|
|2022|  6|            2311|
|2022|  7|            6891|
|2022|  8|            5504|
+----+---+----------------+

+-----------------+
|    dias_promedio|
+-----------------+
|5.022571086312339|
+-----------------+

+--------------------+------------------+
|title               |dias_promedio     |
+--------------------+------------------+
|FLIGHT LIES         |7.333333333333333 |
|IMPACT ALADDIN      |7.333333333333333 |
|AFRICAN EGG         |7.181818181818182 |
|HARDLY ROBBERS      |7.0               |
|MOTHER OLEANDER     |6.857142857142857 |
|HUNTER ALTER        |6.8               |
|MADRE GABLES        |6.785714285714286 |
|SILVERADO GOLDFINGER|6.7272727272727275|
|NOTORIOUS REUNION   |6.714285714285714 |
|SMILE EARRING       |6.7               |
+--------------------+------------------+



#### 3.6 CASO DE NEGOCIO - CLIENTES VIP

```
Identifica a los clientes más valiosos (VIP) aplicando múltiples criterios
de filtro: más de 30 alquileres Y más de $100 gastados.
--join
 customer.join(rental, customer.customer_id == rental.customer_id) \
    .join(payment, rental.rental_id == payment.rental_id) \
--Agrupar por cliente (GROUP BY)
.groupBy(customer.customer_id, customer.first_name, customer.last_name, customer.email)
--Calcular métricas (AGG)
.agg(
    count("*").alias("total_alquileres"),        # ¿Cuántas veces alquiló?
    spark_sum(payment.amount).alias("total_gastado"),  # ¿Cuánto gastó en total?
    avg(payment.amount).alias("promedio_gasto")        # ¿Cuánto gasta en promedio?
)
--Filtrar clientes VIP (FILTER)
.filter(
    (col("total_alquileres") > 30) &   # Cliente frecuente (más de 30 alquileres)
    (col("total_gastado") > 100)       # Cliente que gasta (más de $100)
)
```

In [9]:
print("\n" + "="*60)
print("📌 EJERCICIO 6: CLIENTES VIP (30+ alquileres, $100+ gastado)")
print("="*60)

vip_clientes = customer.join(rental, customer.customer_id == rental.customer_id) \
    .join(payment, rental.rental_id == payment.rental_id) \
    .groupBy(customer.customer_id, customer.first_name, customer.last_name, customer.email) \
    .agg(
        count("*").alias("total_alquileres"),
        spark_sum(payment.amount).alias("total_gastado"),
        avg(payment.amount).alias("promedio_gasto")
    ) \
    .filter(
        (col("total_alquileres") > 30) & 
        (col("total_gastado") > 100)
    ) \
    .orderBy(desc("total_gastado")) \
    .limit(10)

vip_clientes.show(truncate=False)


📌 EJERCICIO 6: CLIENTES VIP (30+ alquileres, $100+ gastado)
+-----------+----------+---------+---------------------------------+----------------+------------------+------------------+
|customer_id|first_name|last_name|email                            |total_alquileres|total_gastado     |promedio_gasto    |
+-----------+----------+---------+---------------------------------+----------------+------------------+------------------+
|526        |KARL      |SEAL     |KARL.SEAL@sakilacustomer.org     |45              |221.55000000000013|4.9233333333333364|
|148        |ELEANOR   |HUNT     |ELEANOR.HUNT@sakilacustomer.org  |46              |216.5400000000001 |4.707391304347828 |
|144        |CLARA     |SHAW     |CLARA.SHAW@sakilacustomer.org    |42              |195.58000000000007|4.656666666666668 |
|137        |RHONDA    |KENNEDY  |RHONDA.KENNEDY@sakilacustomer.org|39              |194.61000000000007|4.990000000000002 |
|178        |MARION    |SNYDER   |MARION.SNYDER@sakilacustomer.org |39 

### 4. Ejercicios practica trabajo

#### 4.1 Análisis de Actores

Contexto: Eres el gerente de un videoclub y quieres conocer más sobre los\
actores que participan en tus películas.

Tarea:

-- Encuentra el Top 5 de actores que han participado en más películas\
-- Muestra el nombre completo del actor y la cantidad de películas

In [35]:
actor.show(2)
film_actor.show(2)
# validar duplicados
film_actor_dup=film_actor.groupBy("actor_id","film_id").agg(count("*").alias("total_rec")).orderBy("total_rec", ascending=True) 
film_actor_dup.show(5)


+--------+----------+---------+-------------------+
|actor_id|first_name|last_name|        last_update|
+--------+----------+---------+-------------------+
|       1|  PENELOPE|  GUINESS|2022-02-15 04:34:33|
|       2|      NICK| WAHLBERG|2022-02-15 04:34:33|
+--------+----------+---------+-------------------+
only showing top 2 rows
+--------+-------+-------------------+
|actor_id|film_id|        last_update|
+--------+-------+-------------------+
|       1|      1|2022-02-15 05:05:03|
|       1|     23|2022-02-15 05:05:03|
+--------+-------+-------------------+
only showing top 2 rows
+--------+-------+---------+
|actor_id|film_id|total_rec|
+--------+-------+---------+
|       1|      1|        1|
|       1|    277|        1|
|       1|    506|        1|
|       1|    832|        1|
|       1|    970|        1|
+--------+-------+---------+
only showing top 5 rows


In [69]:
# join actor y film_actor
res_join=actor.join(film_actor,actor.actor_id==film_actor.actor_id)\
    .select(actor.actor_id, actor.first_name, actor.last_name, film_actor.film_id)\
    .orderBy("actor_id","film_id")
#res_join.show(5)
#res_join.count()
# seleccionando top actores
print('top actores')
top_act=res_join.groupBy("actor_id","first_name","last_name")\
    .agg(count("*").alias("total_peliculas"))\
    .orderBy("total_peliculas", ascending=False)
top_act.show(5)


top actores
+--------+----------+---------+---------------+
|actor_id|first_name|last_name|total_peliculas|
+--------+----------+---------+---------------+
|     107|      GINA|DEGENERES|             42|
|     102|    WALTER|     TORN|             41|
|     198|      MARY|   KEITEL|             40|
|     181|   MATTHEW|   CARREY|             39|
|      23|    SANDRA|   KILMER|             37|
+--------+----------+---------+---------------+
only showing top 5 rows


#### 4.2 Análisis de Duración

Quieres entender cómo se distribuyen las duraciones de tus películas.

Tarea:
1. Clasifica las películas por duración en 3 categorías:\
-- "Corta": menos de 60 minutos\
-- "Media": entre 60 y 119 minutos\
-- "Larga": 120 minutos o más

2. Para cada categoría, calcula:\
-- Duracion total de peliculas\
-- Cantidad de películas\
-- Precio promedio de renta

In [99]:
duracion_films=film.select("title","length")\
    .withColumn("categoria_duracion",
                when(col("length")<60,"Corta")
                .when(col("length")<120,"Media")
                .when(col("length")>=120,"Larga")
                .otherwise("Descononcida"))\
    .orderBy("title",ascending=True)
duracion_films.show(5)
precio_prom=duracion_films.groupBy("categoria_duracion")\
    .agg(spark_sum("length").alias("total_duracion"),
         count("*").alias("total_peliculas"),avg("length").alias("precio_promedio"))
precio_prom.show()

+----------------+------+------------------+
|           title|length|categoria_duracion|
+----------------+------+------------------+
|ACADEMY DINOSAUR|    86|             Media|
|  ACE GOLDFINGER|    48|             Corta|
|ADAPTATION HOLES|    50|             Corta|
|AFFAIR PREJUDICE|   117|             Media|
|     AFRICAN EGG|   130|             Larga|
+----------------+------+------------------+
only showing top 5 rows
+------------------+--------------+---------------+------------------+
|categoria_duracion|total_duracion|total_peliculas|   precio_promedio|
+------------------+--------------+---------------+------------------+
|             Media|         39169|            438|  89.4269406392694|
|             Corta|          5030|             96|52.395833333333336|
|             Larga|         71073|            466|152.51716738197425|
+------------------+--------------+---------------+------------------+



#### 4.3 Análisis de Clientes por Ciudad

Quieres saber qué ciudades tienen los clientes que más gastan.

Tarea:

1. Encuentra el Top 5 de ciudades donde los clientes han gastado más dinero
2. Muestra:\
-- Nombre de la ciudad\
-- Total gastado en esa ciudad\
-- Número de clientes en esa ciudad

In [110]:
ubicacion_join=customer.join(address,customer.address_id==address.address_id)\
    .join(city, address.city_id==city.city_id)\
    .join(payment, customer.customer_id==payment.customer_id)\
    .select(customer.customer_id,customer.first_name,customer.last_name,
            payment.amount,city.city)\
    .groupBy(customer.customer_id,customer.first_name,customer.last_name,city.city)\
    .agg(spark_sum("amount").alias("total_gastado"),count("*").alias("total_transacciones"))
ubicacion_join.show(5)

+-----------+----------+---------+----------+------------------+-------------------+
|customer_id|first_name|last_name|      city|     total_gastado|total_transacciones|
+-----------+----------+---------+----------+------------------+-------------------+
|          6|  JENNIFER|    DAVIS|    Laredo| 93.71999999999996|                 28|
|         98|   LILLIAN|  GRIFFIN|    al-Ayn|106.74999999999997|                 25|
|        137|    RHONDA|  KENNEDY| Apeldoorn| 194.6100000000001|                 39|
|         53|   HEATHER|   MORRIS|Nagareyama|115.69999999999996|                 30|
|         49|     JOYCE|  EDWARDS|     Jedda|130.71999999999997|                 28|
+-----------+----------+---------+----------+------------------+-------------------+
only showing top 5 rows


In [119]:
reporte_ciudad=ubicacion_join.groupBy("city")\
    .agg(spark_sum("total_gastado").alias("total_gastado"),
         spark_sum("total_transacciones").alias("total_transacciones"),
         count("*").alias("total_clientes"))\
    .orderBy("total_gastado", ascending=False)
reporte_ciudad.show(5)

+-----------+------------------+-------------------+--------------+
|       city|     total_gastado|total_transacciones|total_clientes|
+-----------+------------------+-------------------+--------------+
| Cape Coral|221.55000000000013|                 45|             1|
|Saint-Denis| 216.5400000000001|                 46|             1|
|     Aurora|198.49999999999994|                 50|             2|
|  Molodetno|195.58000000000007|                 42|             1|
|  Apeldoorn| 194.6100000000001|                 39|             1|
+-----------+------------------+-------------------+--------------+
only showing top 5 rows


#### 4.4 Análisis de Alquileres por Hora


Quieres optimizar la apertura de tu videoclub y saber a qué hora se alquilan más películas.

Tarea:

1. Extrae la hora del rental_date
2. Agrupa por hora (0-23)
3. Calcula el total de alquileres por hora
4. Muestra las 5 horas con más alquileres

In [24]:
#from pyspark.sql.functions import col, count, sum as spark_sum, avg, desc, when, row_number, rank, dense_rank, datediff, year, month, hour
#from pyspark.sql.functions import round as spark_round

In [6]:
rental.show(10)

+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
|rental_id|        rental_date|inventory_id|customer_id|        return_date|staff_id|        last_update|
+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
|        2|2022-05-24 16:54:33|        1525|        459|2022-05-28 13:40:33|       1|2022-02-15 21:30:53|
|        3|2022-05-24 17:03:39|        1711|        408|2022-06-01 16:12:39|       1|2022-02-15 21:30:53|
|        4|2022-05-24 17:04:41|        2452|        333|2022-06-02 19:43:41|       2|2022-02-15 21:30:53|
|        5|2022-05-24 17:05:21|        2079|        222|2022-06-01 22:33:21|       1|2022-02-15 21:30:53|
|        6|2022-05-24 17:08:07|        2792|        549|2022-05-26 19:32:07|       1|2022-02-15 21:30:53|
|        7|2022-05-24 17:11:53|        3995|        269|2022-05-29 14:34:53|       2|2022-02-15 21:30:53|
|        8|2022-05-24 17:31:46|        2346|  

Traceback (most recent call last):
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


In [13]:
rental_hour=rental.withColumn("hora",hour("rental_date")).select("rental_id","inventory_id","customer_id","rental_date","hora")
rental_hoursum=rental_hour.groupBy("hora").agg(count("*").alias("total_alquileres")).orderBy("total_alquileres",ascending=False)
rental_hoursum.show(24)

+----+----------------+
|hora|total_alquileres|
+----+----------------+
|  10|             846|
|   9|             705|
|   2|             696|
|  18|             694|
|  12|             688|
|  21|             684|
|  22|             681|
|  13|             676|
|   4|             673|
|  15|             671|
|   1|             667|
|   5|             663|
|  14|             658|
|   8|             653|
|   3|             652|
|  19|             649|
|  23|             648|
|   0|             647|
|   7|             645|
|  17|             642|
|  11|             634|
|   6|             632|
|  20|             630|
|  16|             610|
+----+----------------+



#### 4.5 Análisis de Rentabilidad por Actor


Quieres saber qué actores generan más ingresos para tu videoclub.

Tarea:

1. Calcula los ingresos totales generados por cada actor
2. Encuentra el Top 5 de actores que más ingresos generan
3. Muestra:\
-- Nombre completo del actor\
-- Ingresos totales\
-- Número de películas en las que participa

In [16]:
res_join_act_pay=actor.join(film_actor,actor.actor_id==film_actor.actor_id,"left")\
    .join(film, film_actor.film_id==film.film_id,"left")\
    .join(inventory, film.film_id==inventory.film_id,"left")\
    .join(rental, inventory.inventory_id==rental.inventory_id,"left")\
    .join(payment, rental.rental_id==payment.rental_id,"left")\
    .select(actor.actor_id,actor.first_name,actor.last_name,
            film.film_id,film.title,payment.amount)
res_join_act_pay.show(5)

+--------+----------+---------+-------+----------------+------+
|actor_id|first_name|last_name|film_id|           title|amount|
+--------+----------+---------+-------+----------------+------+
|       2|      NICK| WAHLBERG|      3|ADAPTATION HOLES|  2.99|
|       1|  PENELOPE|  GUINESS|      1|ACADEMY DINOSAUR|  0.99|
|       1|  PENELOPE|  GUINESS|      1|ACADEMY DINOSAUR|  0.99|
|      55|       FAY|   KILMER|      8| AIRPORT POLLOCK|  4.99|
|       2|      NICK| WAHLBERG|      3|ADAPTATION HOLES|  2.99|
+--------+----------+---------+-------+----------------+------+
only showing top 5 rows


In [25]:
result_act=res_join_act_pay.groupBy("actor_id","first_name","last_name")\
    .agg(spark_round(spark_sum("amount"), 2).alias("total_ganancias"),count("film_id").alias("total_peliculas"))\
    .orderBy("total_ganancias", ascending=False)
result_act.show(10)

+--------+----------+-----------+---------------+---------------+
|actor_id|first_name|  last_name|total_ganancias|total_peliculas|
+--------+----------+-----------+---------------+---------------+
|     107|      GINA|  DEGENERES|        3442.49|            753|
|     181|   MATTHEW|     CARREY|        2742.19|            680|
|     198|      MARY|     KEITEL|        2689.25|            675|
|      81|  SCARLETT|      DAMON|        2655.28|            573|
|     102|    WALTER|       TORN|        2620.62|            642|
|     144|    ANGELA|WITHERSPOON|        2614.46|            654|
|      58| CHRISTIAN|     AKROYD|        2611.49|            553|
|      60|     HENRY|      BERRY|        2602.88|            612|
|      28|     WOODY|    HOFFMAN|         2546.4|            561|
|     111|   CAMERON|  ZELLWEGER|        2529.41|            563|
+--------+----------+-----------+---------------+---------------+
only showing top 10 rows


#### 4.6 Error del worker

Solucionar el error generado por las particiones

In [3]:
rental.show(5)

+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
|rental_id|        rental_date|inventory_id|customer_id|        return_date|staff_id|        last_update|
+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
|        2|2022-05-24 16:54:33|        1525|        459|2022-05-28 13:40:33|       1|2022-02-15 21:30:53|
|        3|2022-05-24 17:03:39|        1711|        408|2022-06-01 16:12:39|       1|2022-02-15 21:30:53|
|        4|2022-05-24 17:04:41|        2452|        333|2022-06-02 19:43:41|       2|2022-02-15 21:30:53|
|        5|2022-05-24 17:05:21|        2079|        222|2022-06-01 22:33:21|       1|2022-02-15 21:30:53|
|        6|2022-05-24 17:08:07|        2792|        549|2022-05-26 19:32:07|       1|2022-02-15 21:30:53|
+---------+-------------------+------------+-----------+-------------------+--------+-------------------+
only showing top 5 rows


Traceback (most recent call last):
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/opt/spark-4.1.2/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


In [18]:
print(f"\n📊 Particiones actuales: {rental.rdd.getNumPartitions()}")




📊 Particiones actuales: 4
